In [1]:
import cv2
import numpy as np

# Use the callback from the assignment snippet
points = []
def mouse_callback(event, x, y, flags, param):
    global points, img_display
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", img_display)

# Setup
img = cv2.imread("media\\turf.jpg")
flag = cv2.imread("media\\srilanka.png") # Replace with any flag image
img_display = img.copy()

cv2.namedWindow("Image")
cv2.setMouseCallback("Image", mouse_callback)
cv2.imshow("Image", img_display)
cv2.waitKey(0)
cv2.destroyAllWindows()

if len(points) == 4:
    # Source points: Corners of the flag image
    h_f, w_f = flag.shape[:2]
    pts_src = np.array([[0, 0], [w_f - 1, 0], [w_f - 1, h_f - 1], [0, h_f - 1]], dtype=float)
    
    # Destination points: Selected points on turf (TL, TR, BR, BL order expected)
    pts_dst = np.array(points, dtype=float)
    
    # Calculate Homography
    h, status = cv2.findHomography(pts_src, pts_dst)
    
    # Warp flag to turf perspective
    warped_flag = cv2.warpPerspective(flag, h, (img.shape[1], img.shape[0]))
    
    # Create mask and overlay
    mask = np.zeros_like(img, dtype=np.uint8)
    cv2.fillConvexPoly(mask, pts_dst.astype(int), (255, 255, 255))
    
    img_final = np.where(mask == 255, warped_flag, img)
    
    cv2.imshow("Result", img_final)
    cv2.waitKey(0)